In [ ]:
!pip install ipython-autotime fastavro
%load_ext autotime

# Funciones & comprehension

Genera una lista de alumnos aprobados, con nota mayor o igual a 5

In [ ]:
def alumnos_aprobados(estudiantes):
  try:
    result = [x["nombre"] for x in estudiantes if x["nota"] >= 5]
    return result
  except Exception as e:
    print(f"An error occurred: {e}")
    return


estudiantes = [
    {"nombre": "Ana", "nota": 7},
    {"nombre": "Luis", "nota": 4},
    {"nombre": "Carlos", "nota": 9},
    {"nombre": "Marta", "nota": 6}
]

alumnos_aprobados(estudiantes)


Usar la función `es_primo`para generar una diccionario que contenga como clave los números del 1 al 100 y como valor un booleano indicando si es primo o no.

In [ ]:
def es_primo(n):
    return not any( (n % d == 0 for d in range(2, n // 2 + 1)) )

num_primos = {num: es_primo(num) for num in range(1, 101)}

num_primos

Crea una función llamada `es_divisible` que reciba dos argumentos, `x` e `y`, e indique si `x` es divisible por `y` o no (debe devolver un booleano).

Comprueba si `7893209` es divisible por 9. ¿Y por 23?

In [ ]:
def es_divisible(x,y):
    return x % y == 0

¿Qué pasa si dividimos entre 0? ¿Cómo lo resolverías?

In [ ]:
def es_divisible(x,y):
  try:
    return x % y == 0
  except:
    return False

In [ ]:
es_divisible(50,5)

Genera una función que devuelva el cuadrado de los números de un valor inicial a uno final (incluido) usando list comprehension

In [ ]:
def cuadrados(init_number, end_number):
  try:
    result = [x**2 for x in range(init_number, end_number + 1)]
    return result
  except Exception as e:
    print(f"An error occurred: {e}")
    return

cuadrados(1, 10)

# Pandas & Polars

In [ ]:
import pandas as pd
from fastavro import writer, parse_schema

# Supongamos que contamos con un archivo CSV con datos de ventas.
# El CSV contiene columnas: fecha, categoria y valor.
df = pd.read_csv('ventas.csv', sep=";")
print("Datos de ventas (CSV):")
print(df.head())

# Guardar el DataFrame en formato JSON.
# Orient define el tipo de valores del diccionario https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_dict.html
df.to_json('ventas_pd.json', orient='records', lines=True)
print("Se ha exportado el archivo en formato JSON.")

# Guardar el DataFrame en formato Parquet.
df.to_parquet('ventas_pd.parquet')
print("Se ha exportado el archivo en formato Parquet.")

# Exportar a formato Avro.
# Definimos un esquema para los datos.
schema = {
    "doc": "Datos de ventas",
    "name": "Venta",
    "namespace": "analisis.ventas",
    "type": "record",
    "fields": [
        {"name": "fecha", "type": "string"},
        {"name": "categoria", "type": "string"},
        {"name": "valor", "type": "float"}
    ]
}
parsed_schema = parse_schema(schema)
# Convertimos el DataFrame a una lista de diccionarios.
records = df.to_dict(orient='records')
with open('ventas_pd.avro', 'wb') as out:
    writer(out, parsed_schema, records)
print("Se ha exportado el archivo en formato Avro.")

In [ ]:
from os import path
import polars as pl

# Supongamos que contamos con un archivo CSV con datos de ventas.
# El CSV contiene columnas: fecha, categoria y valor.
df = pl.read_csv('ventas.csv',separator=";")
print("Datos de ventas (CSV):")
print(df.head())

# Guardar el DataFrame en formato JSON.
df.write_json('ventas.json')
print("Se ha exportado el archivo en formato JSON.")

# Guardar el DataFrame en formato Parquet.
df.write_parquet('ventas.parquet')
print("Se ha exportado el archivo en formato Parquet.")

# Exportar a formato Avro.
path="ventas.avro"
df.write_avro(path)
print("Se ha exportado el archivo en formato Avro.")

## Leer avro pandas

In [ ]:
import fastavro
import pandas as pd

with open('ventas_pd.avro', 'rb') as f:
    avro_reader = fastavro.reader(f)
    schema = avro_reader.writer_schema
    df_avro = pd.DataFrame(avro_reader)

df_avro.head(15)

## Leer avro Polars

In [ ]:
df = pl.read_avro("ventas.avro")
df

## Operaciones pandas & polars

In [ ]:
import pandas as pd
import polars as pl
print("####### Pandas ######")

# Cargar datos de un archivo CSV con Pandas.
df_pandas = pd.read_csv('ventas.csv', sep=";")
print("\nPrimeras filas de ventas (Pandas):")
print(df_pandas.head())

# Realizar una transformación en Pandas:
# Filtrar ventas con un valor mayor a 100 y calcular el promedio de ventas por categoría.
df_filtrado = df_pandas[df_pandas['valor'] > 100]
promedio_por_categoria = df_filtrado.groupby('categoria')['valor'].mean()
print("\nPromedio de ventas por categoría (Pandas):")
print(promedio_por_categoria)

print("\n####### Polars ######")

# Convertir el DataFrame de Pandas a un DataFrame de Polars.
df_polars = pl.read_csv('ventas.csv', separator=";")
print("\nPrimeras filas de ventas (Polars):")
print(df_polars.head())

# Usar Polars para agrupar y calcular el promedio de ventas por categoría
resultado_polars = (df_polars
                    .filter(pl.col("valor") > 100)
                    .group_by("categoria")
                    .agg(pl.col("valor").mean().alias("promedio")))
print("\nPromedio de ventas por categoría (Polars):")
print(resultado_polars)

## Aplicar funciones pandas & polars

In [ ]:
import pandas as pd
import polars as pl

df_pandas = pd.read_csv('ventas.csv', sep=";")
df_polars = pl.read_csv('ventas.csv', separator=";")

# Definir función auxiliar
def check_expensive(value):
  return "expensive" if value > 75 else "cheap"

df_pandas["expensive"] = df_pandas["valor"].apply(check_expensive)
print("\nPandas\n")
print(df_pandas.head(15))


df_polars = df_polars.with_columns(
    pl.col("valor").map_elements(check_expensive, return_dtype=pl.String).alias("expensive")
)
print("\nPolars\n")
print(df_polars.head(15))

In [ ]:
result = [x for x in range(10) if x%2 == 0]
result